# Conversión de DICOM a BIDS mediante HeudiConv

El estándar **BIDS (Brain Imaging Data Structure)** define una organización jerárquica y reproducible para datos de neuroimagen. La conversión desde formato **DICOM** (formato clínico de origen) hacia BIDS es un paso imprescindible antes de aplicar cualquier pipeline de preprocesado como *fMRIPrep* o *FreeSurfer*.

En este notebook se automatiza dicha conversión utilizando **HeudiConv** ejecutado sobre **Docker**, permitiendo procesar múltiples sujetos de forma sistemática a partir de imágenes T1w de tipo MPRAGE.

## 1. Configuración de rutas

Se definen las rutas de entrada y salida del proceso de conversión:

- `ruta_dicom` — directorio raíz con los archivos DICOM organizados por sujeto  
- `ruta_bids` — directorio de destino donde se generará la estructura BIDS

## 2. Creación del archivo heurístico

HeudiConv requiere un fichero **heurístico** (`.py`) que le indique cómo mapear las series DICOM a la nomenclatura BIDS. En este caso:

- Se define una única clave de destino: `sub-{subject}/anat/sub-{subject}_T1w`  
- La función `infotodict` recorre la información de cada secuencia y filtra aquellas cuya descripción contiene el término `MPRAGE`  
- Se evitan duplicados utilizando `series_id` como identificador único por serie

## 3. Detección automática de sujetos

Se recorre el directorio DICOM para construir la lista de sujetos a procesar. Cada subdirectorio de primer nivel se interpreta como un sujeto distinto. La lista se ordena alfabéticamente para garantizar un procesado reproducible.

## 4. Conversión DICOM → BIDS con HeudiConv (Docker)

Para cada sujeto detectado se lanza un contenedor Docker con la imagen `nipy/heudiconv:latest`. El comando monta tres volúmenes:

| Volumen local | Ruta en contenedor | Propósito |
|---|---|---|
| `ruta_dicom` | /data | Archivos DICOM de entrada |
| `ruta_bids` | /out | Estructura BIDS de salida |
| `ruta_heuristico` | /heuristic.py | Fichero de mapeo |

Los parámetros clave del comando son:

- `-d` — patrón glob para localizar los `.dcm` dentro de la jerarquía de cada sujeto  
- `-c dcm2niix` — motor de conversión a NIfTI  
- `-b` — activa la generación de estructura BIDS  
- `--overwrite` — permite sobrescribir conversiones anteriores

## 5. Limpieza post-conversión

HeudiConv genera automáticamente una serie de archivos y carpetas auxiliares que no forman parte de la estructura BIDS final. Tras completar todas las conversiones se eliminan los siguientes elementos residuales:

| Elemento | Tipo | Motivo de eliminación |
|---|---|---|
| `.heudiconv` | Carpeta | Metadatos internos del proceso |
| `derivatives` | Carpeta | Vacía si no se generaron derivados |
| `.bidsignore` | Archivo | Configuración de ignorados BIDS |
| `CHANGES` | Archivo | Registro de cambios genérico |
| `README` | Archivo | README genérico |

In [2]:
import os
import subprocess
import shutil
ruta_dicom = (
    r"D:\ALZHEIMER3\ADNI2_MCI_F\ADNI"
)  # Ruta de entrada DICOM
ruta_bids = (
    r"D:\ALZHEIMER3\BIDS_MRI_AD_M"
)  # Ruta de salida BIDS
# Crear archivo heuristic (.py) para HeudiConv
ruta_heuristico = os.path.join(ruta_bids, "heuristic.py")
# Contenido del heurístico
heuristico = """
def definir_destino(plantilla, tipo_salida=('nii.gz',)):
    return plantilla, tipo_salida
# Definimos la clave para T1w
t1w = definir_destino('sub-{subject}/anat/sub-{subject}_T1w')
def infotodict(seqinfo):
    info = {t1w: []}
    mejores = {}  # Guardaremos series únicas por nombre de serie
    for s in seqinfo:
        desc = s.series_description.upper()
        is_mprage = "MPRAGE" in desc
        if is_mprage:
            # Evitar duplicados usando series_id como clave
            if s.series_id not in mejores:
                mejores[s.series_id] = s
    # Añadimos todas las series únicas a info
    for s in mejores.values():
        info[t1w].append(s.series_id)
    return info
"""
archivo = open(ruta_heuristico, "w")  # Crear archivo heuristic.py
archivo.write(heuristico)             # Escribir dentro del archivo
archivo.close()                       # Cerrar archivo
print("Heurístico guardado en:", ruta_heuristico)
sujetos = []  # Lista de sujetos
for sujeto in os.listdir(ruta_dicom):
    ruta_sujeto = os.path.join(ruta_dicom, sujeto)
    if os.path.isdir(ruta_sujeto):
        sujetos.append(sujeto)
sujetos.sort()
print("Sujetos encontrados:", len(sujetos))  # Mostrar la cantidad de sujetos encontrados
# Bucle principal: convertir cada sujeto de DICOM a BIDS/NIfTI usando HeudiConv en Docker
for idx_sujeto, suj in enumerate(sujetos, start=1):
    print(f"Procesando: Sujeto {idx_sujeto}...", end=" ", flush=True)
    # Construcción del comando Docker para ejecutar heudiconv
    comando = [
        "docker", "run", "--rm",                     # Ejecutar contenedor y eliminarlo al terminar
        "-v", f"{ruta_dicom}:/data:ro",              # Montar carpeta DICOM como solo lectura
        "-v", f"{ruta_bids}:/out",                   # Montar carpeta de salida BIDS
        "-v", f"{ruta_heuristico}:/heuristic.py",    # Montar el archivo heurístico
        "nipy/heudiconv:latest",                     # Imagen Docker de HeudiConv
        "-d", "/data/{subject}/*/*/*/*.dcm",         # Patrón de búsqueda de archivos DICOM
        "-s", suj,                                   # Nombre del sujeto a procesar
        "-o", "/out",                                # Directorio de salida
        "-f", "/heuristic.py",                       # Archivo heurístico a usar
        "-c", "dcm2niix",                            # Convertidor a utilizar
        "-b",                                        # Construir estructura BIDS
        "--overwrite",                               # Sobrescribir archivos existentes
    ]
    # Ejecutar el comando y capturar la salida
    resultado = subprocess.run(
        comando,
        capture_output=True,
        text=True,
    )
    print("Hecho.")
# Lista de archivos/carpetas generados automáticamente por HeudiConv que no son necesarios
elementos_a_eliminar = [
    ".heudiconv",     # Carpeta oculta con metadatos del proceso
    "derivatives",    # Carpeta de derivados (vacía si no se generaron)
    ".bidsignore",    # Archivo de ignorados de BIDS
    "CHANGES",        # Archivo de cambios/cambios de versión
    "README",         # Archivo README genérico
]
# Limpieza post-conversión: eliminar archivos y carpetas residuales
for elemento in elementos_a_eliminar:
    ruta_elemento = os.path.join(ruta_bids, elemento)
    if os.path.exists(ruta_elemento):               # Verificar si existe
        if os.path.isdir(ruta_elemento):            # Si es directorio
            shutil.rmtree(ruta_elemento)            # Eliminar recursivamente
        else:                                       # Si es archivo
            os.remove(ruta_elemento)                # Eliminar archivo

Heurístico guardado en: D:\ALZHEIMER3\BIDS_MRI_AD_M\heuristic.py
Sujetos encontrados: 23
Procesando: Sujeto 1... Hecho.
Procesando: Sujeto 2... Hecho.
Procesando: Sujeto 3... Hecho.
Procesando: Sujeto 4... Hecho.
Procesando: Sujeto 5... Hecho.
Procesando: Sujeto 6... Hecho.
Procesando: Sujeto 7... Hecho.
Procesando: Sujeto 8... Hecho.
Procesando: Sujeto 9... Hecho.
Procesando: Sujeto 10... Hecho.
Procesando: Sujeto 11... Hecho.
Procesando: Sujeto 12... Hecho.
Procesando: Sujeto 13... Hecho.
Procesando: Sujeto 14... Hecho.
Procesando: Sujeto 15... Hecho.
Procesando: Sujeto 16... Hecho.
Procesando: Sujeto 17... Hecho.
Procesando: Sujeto 18... Hecho.
Procesando: Sujeto 19... Hecho.
Procesando: Sujeto 20... Hecho.
Procesando: Sujeto 21... Hecho.
Procesando: Sujeto 22... Hecho.
Procesando: Sujeto 23... Hecho.
